# Zebra DS457 — SSI barcode scanner

Host-triggered SSI over USB CDC. `connect()` applies all the device settings
(host trigger, continuous power, packeted decode, beeper); then each
**`detect()`** arms the scanner, grabs one barcode, and disarms it.

Pass **`detect(allowed=[...])`** to accept only certain symbologies — the
default accepts every type. Create the scanner with `beep=True` if you want it
to beep on a good read.

## Setup

In [ ]:
from ds457_driver import DS457

# port: "/dev/ttyACM0" (Linux) or "COMx" (Windows)
# beep=True -> scanner beeps on a good read
scanner = DS457(port="/dev/ttyACM0", beep=False)

## Connect

Opens the port **and** confirms the scanner answers over SSI — returns
`True`/`False`. Default `enabled=False` leaves scanning off until `detect()`.

In [ ]:
print("connected:", scanner.connect())

# confirm the key settings landed on the device:
#   138=8 Host trigger | 128=0 Continuous | 238=1 packeted | 159=0 ACK/NAK off
print("settings:", scanner.read_params(138, 128, 238, 159))

## Scan

Present a barcode, run this — it grabs it once and turns the scanner back off.

In [ ]:
r = scanner.detect(timeout=10.0)
print("status   :", r.status)
print("symbology:", r.symbology)
print("data     :", r.data)

## Scan several

Just call `detect()` each time — present a barcode each round. Pass
`allowed=[...]` to accept only certain symbologies (e.g.
`scanner.detect(allowed=["code39", "qrcode"])`).

In [ ]:
for i in range(3):
    r = scanner.detect(timeout=15.0)
    print(f"[{i+1}] {r}")

## Limit to one symbology

Pass `allowed=["ean13"]` so only EAN-13 codes count — any other barcode in
front of the scanner is ignored until an EAN-13 shows up or `timeout` elapses.

In [ ]:
r = scanner.detect(allowed=["ean13"], timeout=15.0)
print("status   :", r.status)
print("symbology:", r.symbology)
print("data     :", r.data)

## Close

In [ ]:
scanner.close()
print("closed")